# Fitting the Single-Diode Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiyush-G/semiconductor-diode-fitting/blob/main/explanations/models/02_single_diode_fitting.ipynb)

**`src/models/fitting.py`**


---

### Contents


|  | | |
|---|---|---|
| **1** | What is a J–V curve? |
| **2** | The equivalent circuit |  
| **3** | The inverse problem |
| **4** | How `fitting.py` is built |
| **5** | **Degeneracy** |
| **6** | Assumptions and gaps |
| **7** | References |

### How to run

`Runtime → Run all` (or ⌘/Ctrl+F9). Takes about a minute.

## 0 · Setup

The cell below finds the repo.

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/Aiyush-G/semiconductor-diode-fitting.git"
IN_COLAB = "google.colab" in sys.modules


def find_root(start: pathlib.Path) -> pathlib.Path | None:
    """Walk up from `start` looking for the repo root (the dir containing src/models)."""
    for p in [start, *start.parents]:
        if (p / "src" / "models" / "fitting.py").exists():
            return p
    return None


root = find_root(pathlib.Path.cwd())
if root is None:
    dest = pathlib.Path("/content" if IN_COLAB else ".") / "semiconductor-diode-fitting"
    if not (dest / "src").exists():
        print(f"Cloning {REPO_URL} -> {dest}")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True
        )
    root = dest

root = root.resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Repository root: {root}")

Cloning https://github.com/Aiyush-G/semiconductor-diode-fitting.git -> /content/semiconductor-diode-fitting
Repository root: /content/semiconductor-diode-fitting


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# The physics + fitting core (numpy/scipy only).
import src.models.fitting as fitting_module
from src.models.examples import DARK_JV_EXAMPLE, LIGHT_JV_EXAMPLE
from src.models.fitting import (
    DEFAULT_BOUNDS,
    PARAM_NAMES,
    default_specs,
    fit_diode,
    pack,
    resolve_residual_space,
)
from src.models.single_diode import (
    DiodeParams,
    key_metrics,
    solve_current,
    thermal_voltage,
)

# The app's own figure builders
from ui.plotting import iv_curve_figure, log_jv_figure, residual_figure

# Colab renders plotly out of the box; this only matters for other frontends.
if IN_COLAB:
    pio.renderers.default = "colab"

# The whole reuse argument above rests on this being true:
assert "streamlit" not in sys.modules, "ui.plotting should not drag in streamlit"

print("fitting.py  :", fitting_module.__file__)
print("numpy       :", np.__version__)
print("no streamlit: OK")

fitting.py  : /content/semiconductor-diode-fitting/src/models/fitting.py
numpy       : 2.0.2
no streamlit: OK


Interactive controls are `ipywidgets` sliders.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

print("interactive widgets:", "available" if HAVE_WIDGETS else "unavailable (static fallback)")

interactive widgets: available


---
## 1 · J–V curve


Shine standardised light on solar cell (1 sun = 100 mW/cm^2, roughly midday),
and then sweep the voltage across its terminals while measuring the current that comes out.

In [ ]:
ds = LIGHT_JV_EXAMPLE
metrics = key_metrics(ds.voltage, ds.current)

fig = iv_curve_figure(
    ds.voltage, ds.current, metrics=metrics, title="J–V curve Example",
)
fig.show()

print(f"{ds.label}: {ds.voltage.size} points, {ds.voltage.min():.3f}–{ds.voltage.max():.3f} V")

Example: Light JV: 64 points, 0.000–0.613 V


### Reading JV Curve

The blue curve is current; the faint
second trace is power ($P = V \times J$), which is what needs to be maximised.

- **$J_{sc}$ - short-circuit current density.** Current at $V = 0$.
- **$V_{oc}$ - open-circuit voltage.** The voltage where current falls to zero. No current
  flows, so no power is delivered.
- **MPP - maximum power point.**
- **Fill factor** - $P_{max} / (J_{sc}V_{oc})$: how *square* the curve is. A perfect cell would hold
  $J_{sc}$ right up to $V_{oc}$ (FF = 1); resistance rounds the knee of the graph.

`key_metrics()` extracts these parameters (see below)

In [ ]:
print(f"Jsc         = {metrics['jsc'] * 1e3:7.2f} mA/cm^2   (current at V = 0)")
print(f"Voc         = {metrics['voc']:7.4f} V        ( current hits zero)")
print(f"Pmax        = {metrics['pmax'] * 1e3:7.2f} mW/cm^2   (at the knee)")
print(f"Fill factor = {metrics['fill_factor']:7.3f}          (square the curve is)")
print(f"Efficiency  = {metrics['efficiency'] * 1e2:7.2f} %        (Pmax / 100 mW/cm^2 of sunlight)")

Jsc         =   36.06 mA/cm^2   (current at V = 0)
Voc         =  0.6127 V        ( current hits zero)
Pmax        =   17.03 mW/cm^2   (at the knee)
Fill factor =   0.771          (square the curve is)
Efficiency  =   17.03 %        (Pmax / 100 mW/cm^2 of sunlight)


---
## 2 · The equivalent circuit


**1.Current source.** Light knocks electrons loose at a steady rate, so the cell generates a
constant photocurrent $J_{ph}$ regardless of voltage:

$$ J = J_{ph} $$

A flat line. Infinite power at high voltage - this is quite incorrect. Therefore:

**2. A diode, in parallel.** A solar cell *is* the pn jn / diode. As voltage is increased,
the junction starts conducting backwards through the cell, reducing the photocurrent. This is
recombination — where carriers anhillate each other. It grows exponentially:

$$ J = J_{ph} - J_0\left(e^{V/nV_t} - 1\right) $$

- $J_0$ - **saturation current density**: how leaky the junction is. Smaller is better. It is tiny
  (~$10^{-11}$) but sits under an exponential, so it dominates $V_{oc}$.
- $n$ - **ideality factor**: *how* recombination happens. $n = 1$ is band-to-band; $n = 2$ means
  defect-mediated (Shockley–Read–Hall). Between 1 and 2 in practice.
- $V_t = k_BT/q$ - **thermal voltage**

Introduce defects into the model:

**3. Shunt resistance $R_{sh}$, in parallel.** Manufacturing defects give current a short-cut
straight across the junction. Ideally, R is infinite.

$$ J = J_{ph} - J_0\left(e^{V/nV_t} - 1\right) - \frac{V}{R_{sh}} $$

**4. A series resistance $R_s$, in series.** It should be small (ideally zero). Junction doesn't see $V$ but rather it sees $V + JR_s$:

$$ J = J_{ph} - J_0\left(e^{\frac{V + JR_s}{nV_t}} - 1\right) - \frac{V + JR_s}{R_{sh}} $$


$J$ is on both sides

---



FUll equation:

$$ J = J_{ph} - J_0\left(e^{\frac{V + JR_s}{nV_t}} - 1\right) - \frac{V + JR_s}{R_{sh}} $$

$J$ appears on the left and inside the exponential on the right. It is implicit.

Thus use the standard closed form (Jain & Kapoor) based on the **Lambert W function** -
the inverse of $w \mapsto we^{w}$, which is exactly the shape "unknown inside an exponential *and*
outside it" produces. `solve_current` computes:

$$ a = \frac{R_sR_{sh}J_0}{nV_t(R_s+R_{sh})}, \qquad
   b = \frac{R_{sh}\big(R_s(J_{ph}+J_0)+V\big)}{nV_t(R_s+R_{sh})} $$

$$ J = \frac{R_{sh}(J_{ph}+J_0) - V}{R_s+R_{sh}} - \frac{nV_t}{R_s}\,W\!\left(a\,e^{b}\right) $$


Note $R_s = 0$  and means can't divide by zero


---
## 3 · Fitting (basically the inverse of the above)


$$ (J_{ph},\, J_0,\, n,\, R_s,\, R_{sh}) \;\longrightarrow\; J(V) $$

But now...

$$ J(V)_{\text{measured}} \;\longrightarrow\; (J_{ph},\, J_0,\, n,\, R_s,\, R_{sh})\;? $$

"How wrong" each guess is needs a number. Use difference at every measured point (the **residual**), square
it, average, square-root - the **RMSE**.

In [1]:
specs = default_specs("light", free={"j_ph", "j_0", "n", "r_s", "r_sh"})
result = fit_diode(DS.voltage, DS.current, TEMP_K, specs, kind="light")

p = result.params
print(f"converged : {result.success}  ({result.message})")
print(f"free      : {', '.join(result.free_names)}\n")
print(f"  Jph  = {p.j_ph * 1e3:.4f} mA/cm^2")
print(f"  J0   = {p.j_0:.4e} A/cm^2")
print(f"  n    = {p.n:.4f}")
print(f"  Rs   = {p.r_s:.4f} Ohm.cm^2")
print(f"  Rsh  = {p.r_sh:.2f} Ohm.cm^2\n")
print(f"  RMSE = {result.rmse * 1e3:.4f} mA/cm^2")
print(f"  R^2  = {result.r_squared:.7f}")

NameError: name 'default_specs' is not defined

R² = 0.99996 but needs validating for meaning

---
## 4 · Explanation of `fitting.py`


Formally, the module minimises a sum of squared residuals over the free parameters $\theta$:

$$ \hat\theta \;=\; \arg\min_{\theta \in [\ell,\,u]} \; \tfrac12\sum_{i=1}^{m} r_i(\theta)^2 $$

^
Find the value of the parameter \theta that makes the model fit the data as well as possible, while keeping \theta between the lower and upper bounds.

Argmin finds the point at which the minimum occurs -> i.e.the parameters within the closed bounds that match the models prediction whilst minimising the mean squared error.




### 4.1 · Free vs fixed parameters

Not obliged to fit all five (see the frontend GUI).

```python
@dataclass(frozen=True)
class ParamSpec:
    name: str      # one of PARAM_NAMES
    free: bool     # True → fitted, False → held exactly
    value: float   # initial guess if free, else the fixed value
    lower: float   # bounds in natural (linear) units
    upper: float
    log: bool      # fitted in log10 space?
```

If you independently know a parameter, fixing it collapses that direction of the search entirely
and makes the rest easier to determine.

In experiment if measured $R_s$, then fix this and then curve is likely to be much better!

In [ ]:
fixed_r_s = 0.4200000000001  # deliberately awkward

specs_fx = default_specs(
    "light", free={"j_ph", "j_0", "n", "r_sh"},   # note: r_s NOT free
    initial={"r_s": fixed_r_s},
)
res_fx = fit_diode(DS.voltage, DS.current, TEMP_K, specs_fx, kind="light")

print(f"fitted      : {', '.join(res_fx.free_names)}")
print(f"Rs supplied : {fixed_r_s!r}")
print(f"Rs returned : {res_fx.params.r_s!r}")
print(f"bitwise identical: {res_fx.params.r_s == fixed_r_s}")

theta0, lower, upper, free_names = pack(specs_fx)
print(f"\noptimiser only ever sees {len(free_names)} numbers: {free_names}")
print("r_s is not among them, so it physically cannot drift.")

fitted      : j_ph, j_0, n, r_sh
Rs supplied : 0.4200000000001
Rs returned : 0.4200000000001
bitwise identical: True

optimiser only ever sees 4 numbers: ('j_ph', 'j_0', 'n', 'r_sh')
r_s is not among them, so it physically cannot drift.


### 4.2 · Why $J_0$ and $R_{sh}$ are fitted in log space

`LOG_PARAMS = {"j_0", "r_sh"}` - these two are fitted as $\log_{10}$ of their value. $J_{ph}$, $n$
and $R_s$ are fitted linearly.

**The intuition.** $J_0$ is "about $10^{-11}$". A step of $10^{-12}$ is problematic when $J_0 = 10^{-13}$. Thus makes sense to step in **ratio** and logarithm makes most sense here.

The Jacobian $\partial J_i/\partial\theta_j$
describe how much the curve moves when nudge each parameter. If one column is $10^{11}$ times bigger
than another, the small one is invisible. Thus, no size of step is suitable for either parameter.

The Jacobian measures how sensitive the predicted JV curve is to each model parameter, with each column showing how a small change in one parameter affects the current at every voltage point. It is used here to compare the conditioning of the optimization problem: if the Jacobian’s columns are more balanced (as when using $\log_{10}(J_0)$ and $\log_{10}(R_{sh}))$, the optimizer can estimate the parameters more accurately and converge more reliably.

In [ ]:
def jacobian_columns(use_log: bool):
    """Central-difference Jacobian columns at a representative optimum."""
    true = DiodeParams(j_ph=0.036, j_0=1e-13, n=1.1, r_s=0.5, r_sh=2500.0, temp_k=298.15)
    vv = np.linspace(0, 0.62, 80)
    base = np.array([true.j_ph, true.j_0, true.n, true.r_s, true.r_sh])
    cols = []
    for i, name in enumerate(PARAM_NAMES):
        lg = use_log and name in ("j_0", "r_sh")
        x = np.log10(base[i]) if lg else base[i]
        h = 1e-6 * max(abs(x), 1.0)
        up, dn = base.copy(), base.copy()
        up[i] = 10 ** (x + h) if lg else x + h
        dn[i] = 10 ** (x - h) if lg else x - h
        f = lambda pv: solve_current(vv, DiodeParams(*pv, temp_k=298.15))
        cols.append((f(up) - f(dn)) / (2 * h))
    return np.array(cols).T


for use_log in (False, True):
    J = jacobian_columns(use_log)
    norms = np.linalg.norm(J, axis=0)
    s = np.linalg.svd(J, compute_uv=False)
    label = "WITH log10 on J0, Rsh" if use_log else "all-linear (no log)"
    print(f"{label}")
    print("   column norms: " + "  ".join(f"{n}={v:.2e}" for n, v in zip(PARAM_NAMES, norms)))
    print(f"   spread (max/min) = {norms.max() / norms.min():.2e}")
    print(f"   cond(J)          = {s.max() / s.min():.2e}\n")

all-linear (no log)
   column norms: j_ph=8.94e+00  j_0=8.65e+04  n=1.93e-02  r_s=1.24e-03  r_sh=5.35e-07
   spread (max/min) = 1.62e+11
   cond(J)          = 4.65e+11

WITH log10 on J0, Rsh
   column norms: j_ph=8.94e+00  j_0=2.20e-03  n=1.93e-02  r_s=1.24e-03  r_sh=3.08e-03
   spread (max/min) = 7.23e+03
   cond(J)          = 6.74e+06



Not everything is logged:

- **$R_s$ can  be zero**, and $\log(0) = -\infty$. §2 proved the model is perfectly
  continuous through $R_s = 0$/
- **$n$ spans ~1–2** and **$J_{ph}$ is known to a few percent** from $J_{sc}$.

### 4.3 · Bounds, and a one-line clamp that prevents a crash

Currently implemented as hard-boundaries and there isn't a soft penalty term, so this should be explored in more detail later.

In [ ]:
print("DEFAULT_BOUNDS (natural units)      UI slider range        comment")
ui_ranges = {
    "j_ph": ("0 – 50 mA/cm²", "bound is 2x the slider top"),
    "j_0": ("1e-16 – 1e-5", "generous; unlikely to bind"),
    "n": ("1.0 – 2.0", "<-- bound allows n < 1 (unphysical!)"),
    "r_s": ("0 – 5", "zero is legitimate"),
    "r_sh": ("100 – 1e5", "upper bound ~ 'no shunt at all'"),
}
for name in PARAM_NAMES:
    lo, hi = DEFAULT_BOUNDS[name]
    ui, note = ui_ranges[name]
    print(f"  {name:5s} ({lo:g}, {hi:g})".ljust(38) + f"{ui:22s} {note}")

DEFAULT_BOUNDS (natural units)      UI slider range        comment
  j_ph  (0, 0.1)                      0 – 50 mA/cm²          bound is 2x the slider top
  j_0   (1e-20, 0.001)                1e-16 – 1e-5           generous; unlikely to bind
  n     (0.8, 5)                      1.0 – 2.0              <-- bound allows n < 1 (unphysical!)
  r_s   (0, 50)                       0 – 5                  zero is legitimate
  r_sh  (10, 1e+08)                   100 – 1e5              upper bound ~ 'no shunt at all'




```
# This is formatted as code
```

Note (clips each parameter so that it lies within the correct range):
```python
theta0_arr = np.clip(np.asarray(theta0, dtype=float), lower_arr, upper_arr)
```

`least_squares` raises `ValueError` if the starting point $x_0$ is outside the bounds. This would clash with UI so overriden.

In [ ]:
# Seed j_0 at 1e-2, far above its upper bound of 1e-3.
specs_bad = default_specs("light", free={"j_0", "n"}, initial={"j_0": 1e-2})
theta0, lower, upper, names = pack(specs_bad)

print(f"free parameters      : {names}")
print(f"seed j_0 requested   : 1e-02  (log10 = {np.log10(1e-2):+.3f})")
print(f"upper bound on j_0   : {DEFAULT_BOUNDS['j_0'][1]:.0e}  (log10 = {upper[0]:+.3f})")
print(f"seed after clamping  : log10 = {theta0[0]:+.3f}  -> j_0 = {10 ** theta0[0]:.1e}")
print(f"\nstart is now feasible: {bool(np.all((theta0 >= lower) & (theta0 <= upper)))}")
print("Without the clamp this exact call would raise ValueError.")

free parameters      : ('j_0', 'n')
seed j_0 requested   : 1e-02  (log10 = -2.000)
upper bound on j_0   : 1e-03  (log10 = -3.000)
seed after clamping  : log10 = -3.000  -> j_0 = 1.0e-03

start is now feasible: True
Without the clamp this exact call would raise ValueError.


### 4.4 · Linear vs log residuals

The residual is "how wrong is the model at point $i$". There are two sensible definitions and
they are not interchangeable:

$$ r_i^{\text{lin}} = J^{\text{model}}_i - J^{\text{meas}}_i
   \qquad\qquad
   r_i^{\text{log}} = \log_{10}\!|J^{\text{model}}_i| - \log_{10}\!|J^{\text{meas}}_i| $$

`resolve_residual_space("auto", kind)` picks **log for dark data** and **linear for light**.

**Why light data must be linear.** A light curve crosses zero at $V_{oc}$. Log not valid.

In [ ]:
dk = DARK_JV_EXAMPLE
mag = np.abs(dk.current)
print(f"{dk.label}: {dk.voltage.size} points")
print(f"  |J| spans {mag.min():.3e} to {mag.max():.3e} A/cm^2")
print(f"  that is {np.log10(mag.max() / mag.min()):.1f} DECADES of current")
print(f"\nUnder LINEAR residuals, a 1% error at the top of the curve costs")
print(f"~{mag.max() / mag.min() * 0.01:,.0f}x more than a 100% error at the bottom.")
print("The fit would become a fit to the last half-decade, throwing away the")
print("low-injection region -- exactly where shunt and recombination physics live.")

Example: Dark JV: 127 points
  |J| spans 2.347e-06 to 2.463e-01 A/cm^2
  that is 5.0 DECADES of current

Under LINEAR residuals, a 1% error at the top of the curve costs
~1,050x more than a 100% error at the bottom.
The fit would become a fit to the last half-decade, throwing away the
low-injection region -- exactly where shunt and recombination physics live.


In [ ]:
def fit_dark(space):
    sp = default_specs("dark", free={"j_0", "n", "r_s", "r_sh"})
    return fit_diode(dk.voltage, dk.current, 298.15, sp, kind="dark", residual_space=space)


r_log, r_lin = fit_dark("log"), fit_dark("linear")

print(f"{'':16s} {'J0':>12s} {'n':>7s} {'Rs':>7s} {'Rsh':>8s} {'linear R2':>11s} {'worst log err':>15s}")
for tag, r in (("log space", r_log), ("linear space", r_lin)):
    p = r.params
    ld = np.log10(np.maximum(np.abs(r.model_current), 1e-9)) - np.log10(np.abs(dk.current))
    print(f"  {tag:14s} {p.j_0:12.3e} {p.n:7.3f} {p.r_s:7.3f} {p.r_sh:8.1f} "
          f"{r.r_squared:11.5f} {np.max(np.abs(ld)):11.3f} dec")

k = int(np.argmax(np.abs(np.log10(np.maximum(np.abs(r_lin.model_current), 1e-9))
                        - np.log10(np.abs(dk.current)))))
print(f"\nWorst point of the linear-space fit: V = {dk.voltage[k]:.4f} V")
print(f"  measured {dk.current[k]:.3e}   model {r_lin.model_current[k]:.3e} A/cm^2")
print(f"  -> wrong by a factor of {abs(r_lin.model_current[k] / dk.current[k]):.2f}")

                           J0       n      Rs      Rsh   linear R2   worst log err
  log space         3.717e-11   1.172   0.192    439.4     0.99962       0.019 dec
  linear space      2.682e-10   1.298   0.154    738.2     0.99997       0.245 dec

Worst point of the linear-space fit: V = 0.0013 V
  measured -3.093e-06   model -1.761e-06 A/cm^2
  -> wrong by a factor of 0.57


**Above result is misleading and actually gives wrong result.** The linear-space fit has the **better R²**
(0.99997 vs 0.99962). It is nonetheless wrong by 0.245 decades - a factor of 1.76 - at low voltage.

Plot on semi-log for reasoning:

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(dk.current), mode="markers",
                         name="measured (5 decades)", marker=dict(color="#444", size=4)))
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(r_log.model_current), mode="lines",
                         name=f"log-space fit — rmse_log = {r_log.rmse_log:.3f} dec",
                         line=dict(color="#2563eb", width=2.5)))
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(r_lin.model_current), mode="lines",
                         name=f"linear-space fit — R² = {r_lin.r_squared:.5f}",
                         line=dict(color="#dc2626", width=2.5, dash="dash")))
fig.update_layout(
    title="Diff residuals",
    xaxis_title="Voltage (V)", yaxis_title="|J| (A/cm²)",
    yaxis=dict(type="log"), height=460, margin=dict(l=60, r=30, t=60, b=50),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
)
fig.show()

The red curve tracks the high-current region which means it gets R² = 0.99997 — then
 away below ~$10^{-3}$ A/cm², where four of the five decades of measurement are.


In [ ]:
residual_figure(dk.voltage, r_log.residual, residual_space=r_log.residual_space).show()
print("Structureless scatter about zero = good fit.")
print("Systematic curvature would mean the model itself is wrong (see §6, A1).")

Structureless scatter about zero = good fit.
Systematic curvature would mean the model itself is wrong (see §6, A1).


### 4.5 · Dark data should not recieve a photocurrent!

A dark measurement is taken with the lights off, so $J_{ph} = 0$ **by definition**. Fitting it would
mean fitting a quantity known in advance to be zero.

In [ ]:
sneaky = default_specs("dark", free={"j_ph", "j_0", "n"})  # explicitly demanding j_ph
s = sneaky["j_ph"]
print("Asked to fit j_ph on dark data. Got:")
print(f"  free   = {s.free}")
print(f"  value  = {s.value}")
print(f"  bounds = ({s.lower}, {s.upper})")

r = fit_diode(dk.voltage, dk.current, 298.15, sneaky, kind="dark")
print(f"\nactually fitted : {r.free_names}")
print(f"j_ph in result  : {r.params.j_ph}  <- never introduced")

Asked to fit j_ph on dark data. Got:
  free   = False
  value  = 0.0
  bounds = (0.0, 0.0)

actually fitted : ('j_0', 'n')
j_ph in result  : 0.0  <- never introduced


### 4.6 · The optimiser call

```python
result = least_squares(
    _make_residual(voltage, current, specs, temp_k, space, penalty),
    theta0, bounds=(lower, upper),
    method="trf", x_scale="jac", loss=loss, max_nfev=max_nfev,
)
```

Had to look at documentation to decide the above^

1. Calls SciPy’s nonlinear least-squares optimizer and stores the result in `result`.

2. `make_residual` creates the function that the optimiser will repeatedly evaluate - the aim is to get the residual to close to zero as possible!

The different parameters:

1. TRF stands for Trust Region Reflective - builds a local approximation of the residuals, chooses a step that it “trusts” will improve the fit.

2. `x_scale="jac"` - estimated parameter step based on the Jacobian (see explanation above)

3. Loss function - least squares

`result.x` holds the fitted parameters.

In [ ]:
best = result.params   # the §3 fit
all_fixed = default_specs(
    "light", free=set(),
    initial={"j_ph": best.j_ph, "j_0": best.j_0, "n": best.n,
             "r_s": best.r_s, "r_sh": best.r_sh},
)
r_fixed = fit_diode(DS.voltage, DS.current, TEMP_K, all_fixed, kind="light")
print(f"free_names : {r_fixed.free_names}   (nothing to optimise)")
print(f"success    : {r_fixed.success}")
print(f"message    : {r_fixed.message}")
print(f"RMSE       : {r_fixed.rmse * 1e3:.4f} mA/cm^2  <- still scored against the data")

free_names : ()   (nothing to optimise)
success    : True
message    : No free parameters; evaluated fixed model.
RMSE       : 0.0452 mA/cm^2  <- still scored against the data


### 4.7 · The penalty is a crash guard, not a barrier

If the optimiser creates a set of parameters that cannot be solved in the diode equation then gets returned a really large value so it doesn't fit around something unphysical. The jacobian also evalutes to zero around here so doesn't actually steer the fitting back to a particular value.

In [ ]:
v5 = np.linspace(0.01, 0.6, 20)
j5 = np.full_like(v5, -1e-4)
sp5 = default_specs("dark", free={"j_0", "n"})

original = fitting_module.solve_current
try:
    fitting_module.solve_current = lambda *a, **k: (_ for _ in ()).throw(OverflowError("boom"))
    res_fn = fitting_module._make_residual(v5, j5, sp5, 298.15, "log", fitting_module.PENALTY)
    th, *_ = pack(sp5)
    a = res_fn(th)
    b = res_fn(th + np.array([0.5, 0.2]))
finally:
    fitting_module.solve_current = original   # always restore

print(f"residual at theta0        : {a[:3]}")
print(f"residual at theta0 + delta: {b[:3]}")
print(f"bitwise identical         : {np.array_equal(a, b)}")
print("\n=> the finite-difference Jacobian here is exactly ZERO.")
print("   The penalty stops a crash; it cannot steer the optimiser back to safety.")
print("   The BOUNDS (4.3) are what actually keep it out of trouble.")

residual at theta0        : [1000000. 1000000. 1000000.]
residual at theta0 + delta: [1000000. 1000000. 1000000.]
bitwise identical         : True

=> the finite-difference Jacobian here is exactly ZERO.
   The penalty stops a crash; it cannot steer the optimiser back to safety.
   The BOUNDS (4.3) are what actually keep it out of trouble.


Note on degen

>
> A single-diode fit to a light J–V curve cannot determine $n$ and $J_0$ separately, no matter
> how good the optimiser or how high the R². It determines $J_{ph}$, $V_{oc}$, and *one combination*
> of $n$ and $J_0$.
> Likely need light and dark curve

---
## 7 · References

**Method / literature**

- Jain & Kapoor, *Exact analytical solutions of the parameters of real solar cells using Lambert
  W-function*, Sol. Energy Mater. Sol. Cells — <https://doi.org/10.1016/j.solmat.2003.11.018>
- <https://doi.org/10.1016/j.solmat.2005.04.023>
- <https://doi.org/10.1016/j.rser.2015.11.051>
- <https://doi.org/10.1016/j.solener.2023.111930>
- <https://doi.org/10.1016/j.solener.2023.01.009>
- De Soto et al. (2006) five-parameter model —
  [Sandia PVPMC](https://pvpmc.sandia.gov/modeling-guide/2-dc-module-iv/single-diode-equivalent-circuit-models/de-soto-five-parameter-module-model/)
- [PV Lighthouse equivalent-circuit calculator](https://www.pvlighthouse.com.au/equivalent-circuit) —
  the source of the unit convention and the cross-check reference
- [PVsyst one-diode model](https://www.pvsyst.com/help/physical-models-used/pv-module-standard-one-diode-model/index.html)

